--- Day 10: Factory ---
Just across the hall, you find a large factory. Fortunately, the Elves here have plenty of time to decorate. Unfortunately, it's because the factory machines are all offline, and none of the Elves can figure out the initialization procedure.

The Elves do have the manual for the machines, but the section detailing the initialization procedure was eaten by a Shiba Inu. All that remains of the manual are some indicator light diagrams, button wiring schematics, and joltage requirements for each machine.

For example:

[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
[...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}
[.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}
The manual describes one machine per line. Each line contains a single indicator light diagram in [square brackets], one or more button wiring schematics in (parentheses), and joltage requirements in {curly braces}.

To start a machine, its indicator lights must match those shown in the diagram, where . means off and # means on. The machine has the number of indicator lights shown, but its indicator lights are all initially off.

So, an indicator light diagram like [.##.] means that the machine has four indicator lights which are initially off and that the goal is to simultaneously configure the first light to be off, the second light to be on, the third to be on, and the fourth to be off.

You can toggle the state of indicator lights by pushing any of the listed buttons. Each button lists which indicator lights it toggles, where 0 means the first light, 1 means the second light, and so on. When you push a button, each listed indicator light either turns on (if it was off) or turns off (if it was on). You have to push each button an integer number of times; there's no such thing as "0.5 presses" (nor can you push a button a negative number of times).

So, a button wiring schematic like (0,3,4) means that each time you push that button, the first, fourth, and fifth indicator lights would all toggle between on and off. If the indicator lights were [#.....], pushing the button would change them to be [...##.] instead.

Because none of the machines are running, the joltage requirements are irrelevant and can be safely ignored.

You can push each button as many times as you like. However, to save on time, you will need to determine the fewest total presses required to correctly configure all indicator lights for all machines in your list.

There are a few ways to correctly configure the first machine:

[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
You could press the first three buttons once each, a total of 3 button presses.
You could press (1,3) once, (2,3) once, and (0,1) twice, a total of 4 button presses.
You could press all of the buttons except (1,3) once each, a total of 5 button presses.
However, the fewest button presses required is 2. One way to do this is by pressing the last two buttons ((0,2) and (0,1)) once each.

The second machine can be configured with as few as 3 button presses:

[...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}
One way to achieve this is by pressing the last three buttons ((0,4), (0,1,2), and (1,2,3,4)) once each.

The third machine has a total of six indicator lights that need to be configured correctly:

[.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}
The fewest presses required to correctly configure it is 2; one way to do this is by pressing buttons (0,3,4) and (0,1,2,4,5) once each.

So, the fewest button presses required to correctly configure the indicator lights on all of the machines is 2 + 3 + 2 = 7.

Analyze each machine's indicator light diagram and button wiring schematics. What is the fewest button presses required to correctly configure the indicator lights on all of the machines?

Since all examples in the description had solutions with relatively few button presses, I'm gonna code a program that (1) iterates through all possibilities exhaustively with one, then 2, then 3 button presses and so on and (2) checks at each point if we have reached the correct configuration

In [1]:
import re
with open('input.txt') as file:
    problem_sets = []
    for line in file:
        solution = re.findall(r'[.#]+', line)[0]
        solution = tuple(int(char) for char in solution.replace('.', '0').replace('#', '1'))
        buttons = tuple(
                        tuple(int(number) for number in button_str.split(',')) # turns '(0,1)' into (0, 1)
                        for button_str in re.findall(r'\((.+?)\)', line)) # spits out individual button strings, eg '(0,1)
        # omit joltage
        problem_sets.append((buttons, solution))
problem_sets[5], len(problem_sets)

((((1, 6),
   (0, 2, 5),
   (0, 2, 3, 7),
   (0, 3, 5, 6, 7),
   (0, 1, 2, 5),
   (0, 1, 2, 3, 4, 5, 6),
   (0, 1, 3, 5, 6),
   (0, 1, 4, 5, 6, 7),
   (0, 1, 2, 3, 4, 6)),
  (0, 1, 0, 0, 0, 1, 1, 1)),
 167)

In [2]:
# Regex testing:
line = '[..##.#.] (0,4) (1,3,5) (0,6) (0,2,3,5) (2,5,6) (2,3,5) (0,1,2,4) (0,5) {67,29,30,40,18,54,21}'
print(re.findall(r'[.#]+', line)[0])
print(re.findall(r'\((.+?)\)', line))
print(re.findall(r'\{.+\}', line))

solution = '..##.#.'
tuple(int(char) for char in solution.replace('.', '0').replace('#', '1'))
        

..##.#.
['0,4', '1,3,5', '0,6', '0,2,3,5', '2,5,6', '2,3,5', '0,1,2,4', '0,5']
['{67,29,30,40,18,54,21}']


(0, 0, 1, 1, 0, 1, 0)

In [3]:
# we map solution/state as showing the chars that need to flip as # and those that need to stay as '.'. Therefore, initially, state == solution, while each button press flips affected bits

def press(button, state):
    state = list(state).copy()
    for wire in button:
        state[wire] = 1 if state[wire] == 0 else 0
    return tuple(state)


In [4]:
from collections import deque
total_button_presses = 0
for buttons, solution in problem_sets:
    found_solution = False
    visited = {solution}
    queue = deque()
    queue.append((solution, 1))
    while not found_solution:
        old_state, n = queue.popleft()
        for button in buttons:
            new_state = press(button, old_state)
            if new_state in visited:
                continue
            elif all(i == 0 for i in new_state):
                found_solution = True
                break
            else:
                queue.append((new_state, n + 1))
                visited.add(new_state)
    total_button_presses += n

total_button_presses 

455

Your puzzle answer was 455.

The first half of this puzzle is complete! It provides one gold star: *

--- Part Two ---
All of the machines are starting to come online! Now, it's time to worry about the joltage requirements.

Each machine needs to be configured to exactly the specified joltage levels to function properly. Below the buttons on each machine is a big lever that you can use to switch the buttons from configuring the indicator lights to increasing the joltage levels. (Ignore the indicator light diagrams.)

The machines each have a set of numeric counters tracking its joltage levels, one counter per joltage requirement. The counters are all initially set to zero.

So, joltage requirements like {3,5,4,7} mean that the machine has four counters which are initially 0 and that the goal is to simultaneously configure the first counter to be 3, the second counter to be 5, the third to be 4, and the fourth to be 7.

The button wiring schematics are still relevant: in this new joltage configuration mode, each button now indicates which counters it affects, where 0 means the first counter, 1 means the second counter, and so on. When you push a button, each listed counter is increased by 1.

So, a button wiring schematic like (1,3) means that each time you push that button, the second and fourth counters would each increase by 1. If the current joltage levels were {0,1,2,3}, pushing the button would change them to be {0,2,2,4}.

You can push each button as many times as you like. However, your finger is getting sore from all the button pushing, and so you will need to determine the fewest total presses required to correctly configure each machine's joltage level counters to match the specified joltage requirements.

Consider again the example from before:

[.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
[...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}
[.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}
Configuring the first machine's counters requires a minimum of 10 button presses. One way to do this is by pressing (3) once, (1,3) three times, (2,3) three times, (0,2) once, and (0,1) twice.

Configuring the second machine's counters requires a minimum of 12 button presses. One way to do this is by pressing (0,2,3,4) twice, (2,3) five times, and (0,1,2) five times.

Configuring the third machine's counters requires a minimum of 11 button presses. One way to do this is by pressing (0,1,2,3,4) five times, (0,1,2,4,5) five times, and (1,2) once.

So, the fewest button presses required to correctly configure the joltage level counters on all of the machines is 10 + 12 + 11 = 33.

Analyze each machine's joltage requirements and button wiring schematics. What is the fewest button presses required to correctly configure the joltage level counters on all of the machines?

In [5]:
import re
with open('input.txt') as file:
    problem_sets = []
    for line in file:
        solution = re.findall(r'\{(.+)\}', line)[0]
        solution = tuple(int(number) for number in solution.split(','))
        buttons = tuple(
                        tuple(int(number) for number in button_str.split(',')) # turns '(0,1)' into (0, 1)
                        for button_str in re.findall(r'\((.+?)\)', line)) # spits out individual button strings, eg '(0,1)
        problem_sets.append((buttons, solution))
problem_sets[0], len(problem_sets)

((((0, 4),
   (1, 3, 5),
   (0, 6),
   (0, 2, 3, 5),
   (2, 5, 6),
   (2, 3, 5),
   (0, 1, 2, 4),
   (0, 5)),
  (67, 29, 30, 40, 18, 54, 21)),
 167)

In [6]:
def jolt_it(button, state):
    state = list(state).copy()
    for wire in button:
        state[wire] -= 1
    return tuple(state)

In [7]:
jolt_it((4, 1), (67, 29, 30, 40, 18, 54, 21))

(67, 28, 30, 40, 17, 54, 21)

In [8]:
# total_button_presses = 0
# counter = 0
# for buttons, solution in problem_sets:
#     found_solution = False
#     visited = {solution}
#     queue = deque()
#     queue.append((solution, 1))
#     while not found_solution:
#         old_state, n = queue.popleft()
#         for button in buttons:
#             new_state = jolt_it(button, old_state)
#             if new_state in visited:
#                 continue
#             elif any(i < 0 for i in new_state):
#                 continue
#             elif all(i == 0 for i in new_state):
#                 found_solution = True
#                 break
#             else:
#                 queue.append((new_state, n + 1))
#                 visited.add(new_state)
#     total_button_presses += n
#     counter += 1
#     if counter % 10 == 0:
#         print('.', end='')

# total_button_presses 

In [ ]:
# Too slow! New idea to restrict the search space: kick out attempts that would deviate from a roughly linear path towards the solution. That is, all joltages should be decreasing roughly proportionally, i.e. if the solution is {60, 30, 50}, with sum total 140, the first parameter should stay close to 60/140, second to 30/140, third 50/140 of the sum_total. close might be +- 3


    
def find_solution(buttons, solution, tol=1):
    visited = {solution}
    proportions = tuple(target / sum(solution) for target in solution)
    queue = deque()
    queue.append((solution, 1))

    def knockout_conditions(state, tol=1):
        sum_state = sum(state)
        if state in visited:
            return True
        elif any(i < 0 for i in state):
            return True
        elif not all(target_prop * sum_state - tol < i < target_prop * sum_state + tol
                    for i, target_prop in zip(state, proportions)):
            return True
        else:
            return False

    while True:
        try:
            old_state, n = queue.popleft()
        except IndexError:
            if tol == 1:
                return find_solution(buttons, solution, tol=2)
            else:
                raise IndexError
        for button in buttons:
            new_state = jolt_it(button, old_state)
            if knockout_conditions(new_state, tol):
                continue
            elif all(i == 0 for i in new_state):
                return n
            else:
                queue.append((new_state, n + 1))
                visited.add(new_state)

In [23]:
total_button_presses = 0
counter = 0
for buttons, solution in problem_sets:
    total_button_presses += find_solution(buttons, solution)
    counter += 1
    if counter % 10 == 0:
        print('.', end='')
total_button_presses

................

16978

In [ ]:
That's the right answer! You are one gold star closer to decorating the North Pole.

You have completed Day 10! You can [Share] this victory or [Return to Your Advent Calendar].